In [ ]:
import jax
import jax.numpy as jnp
from evosax.problems import BBOBProblem
from evosax.algorithms import SimpleGA

# 1. Setup JAX random keys
seed = 0
key = jax.random.key(seed)

# 2. Define the Problem (Rosenbrock function)
num_dims = 2
problem = BBOBProblem(
    fn_name="rosenbrock",
    num_dims=num_dims,
    x_opt=2.5 * jnp.ones(num_dims),
    f_opt=0.0,
    sample_rotations=False,
    seed=seed,
)

# 3. Instantiate the Simple GA strategy
population_size = 16
num_generations = 64

key, subkey = jax.random.split(key)
solution = problem.sample(subkey)  # Dummy solution to infer shapes

ga = SimpleGA(
    population_size=population_size,
    solution=solution,
)

# Fetch default parameters
params = ga.default_params

# 4. Get Initial Population and Evaluate It (Required for Population-based algos)
key, subkey = jax.random.split(key)
keys = jax.random.split(subkey, population_size)
population_init = jax.vmap(problem.sample)(keys)

key, subkey = jax.random.split(key)
problem_state = problem.init(subkey)
fitness_init, problem_state, _ = problem.eval(
    key, population_init, problem_state
)

# 5. Initialize the state of the Genetic Algorithm
key, subkey = jax.random.split(key)
state = ga.init(subkey, population_init, fitness_init, params)

# 6. Run the Ask-Eval-Tell optimization loop
print("Starting optimization...")
metrics_log = []

for i in range(num_generations):
    key, key_ask, key_tell = jax.random.split(key, 3)

    # Ask: Generate a set of candidate solutions to evaluate
    population, state = ga.ask(key_ask, state, params)

    # Eval: Evaluate the fitness of the generated population
    fitness, problem_state, _ = problem.eval(
        key_tell, population, problem_state
    )

    # Tell: Update the evolution strategy's state based on the feedback
    state, metrics = ga.tell(key_tell, population, fitness, state, params)

    metrics_log.append(metrics["best_fitness"])

# 7. Output the results
print(f"Optimization complete after {num_generations} generations.")
print(f"Best Fitness Achieved: {metrics['best_fitness']}")
# We can use the base API method from your tests to cleanly extract the best solution
print(f"Best Solution Found: {ga.get_best_solution(state)}")

Starting optimization...
Optimization complete after 64 generations.
Best Fitness Achieved: 0.016836438328027725
Best Solution Found: [0.34289888 2.725772  ]


In [22]:
"""Evosax strategy adapter for the BenchmarkRunner.Engine protocol.

Wraps any evosax population-based strategy (ask/tell interface) so it can
be used interchangeably with GeneticEngineAdapter through Composer.quick_run().

Usage::

    from malthusjax.core.fitness.bbob_evaluator import BBOBEvaluator, BBOBConfig

    # build a MalthusJAX evaluator (could be any BaseEvaluator)
    evalr = BBOBEvaluator.create(BBOBConfig(fn_name="sphere", num_dims=10, seed=0))
    adapter = build_evosax_engine(
        strategy_name="SimpleGA",
        evaluator=evalr,
        pop_size=100,
        generations=200,
    )
    result = adapter.run_once(jax.random.PRNGKey(42))
"""

from __future__ import annotations

import time
from typing import Any, Dict, Optional, Tuple

import chex
import jax
import jax.numpy as jnp
import jax.random as jr
from evosax.algorithms import MR15_GA, DifferentialEvolution, SimpleGA
import evosax
from malthusjax.core.fitness.base import BaseEvaluator
from malthusjax.core.fitness.bbob_evaluator import BBOBConfig, BBOBEvaluator

EVOSAX_STRATEGIES: Dict[str, type] = {
    "SimpleGA": SimpleGA,
    "MR15_GA": MR15_GA,
    "DifferentialEvolution": DifferentialEvolution,
}


def list_strategies() -> list[str]:
    """Return available evosax strategy names."""
    return sorted(EVOSAX_STRATEGIES.keys())

class EvosaxEngineAdapter:
    """Adapter to make evosax strategies compatible with BenchmarkRunner.Engine protocol.

    Implements the same ``run_once(key) -> Dict`` contract as
    :class:`GeneticEngineAdapter` so both can be used interchangeably
    with :class:`BenchmarkRunner`.
    """

    def __init__(
        self,
        strategy: evosax.algorithms.population_based.PopulationBasedAlgorithm,
        params: Any,  # struct.dataclass with strategy hyper-parameters
        problem: evosax.problems.problem.Problem,
        problem_state: Any, # struct.dataclass with problem state
        pop_size: int,
        num_generations: int,
        num_dims: int,
        bounds: Tuple[float, float] = (-5.0, 5.0),
        maximize: bool = False,
        initial_population: chex.Array = None,
        prng_impl: Optional[str] = None,
    ) -> None:
        self.strategy = strategy
        self.params = params
        self.problem = problem
        self.problem_state = problem_state
        self.pop_size = pop_size
        self.num_generations = num_generations
        self.num_dims = num_dims
        self.bounds = bounds
        self.maximize = maximize
        self.initial_population = initial_population
        self.prng_impl = prng_impl

"""Evosax strategy adapter for the BenchmarkRunner.Engine protocol.

Wraps any evosax population-based strategy (ask/tell interface) so it can
be used interchangeably with GeneticEngineAdapter through Composer.quick_run().

Usage::

    from malthusjax.core.fitness.bbob_evaluator import BBOBEvaluator, BBOBConfig

    # build a MalthusJAX evaluator (could be any BaseEvaluator)
    evalr = BBOBEvaluator.create(BBOBConfig(fn_name="sphere", num_dims=10, seed=0))
    adapter = build_evosax_engine(
        strategy_name="SimpleGA",
        evaluator=evalr,
        pop_size=100,
        generations=200,
    )
    result = adapter.run_once(jax.random.PRNGKey(42))
"""

from __future__ import annotations

import time
from typing import Any, Dict, Optional, Tuple

import chex
import jax
import jax.numpy as jnp
import jax.random as jr
from evosax.algorithms import MR15_GA, DifferentialEvolution, SimpleGA
import evosax
from malthusjax.core.fitness.base import BaseEvaluator
from malthusjax.core.fitness.bbob_evaluator import BBOBConfig, BBOBEvaluator

EVOSAX_STRATEGIES: Dict[str, type] = {
    "SimpleGA": SimpleGA,
    "MR15_GA": MR15_GA,
    "DifferentialEvolution": DifferentialEvolution,
}


def list_strategies() -> list[str]:
    """Return available evosax strategy names."""
    return sorted(EVOSAX_STRATEGIES.keys())

class EvosaxEngineAdapter:
    """Adapter to make evosax strategies compatible with BenchmarkRunner.Engine protocol.

    Implements the same ``run_once(key) -> Dict`` contract as
    :class:`GeneticEngineAdapter` so both can be used interchangeably
    with :class:`BenchmarkRunner`.
    """

    def __init__(
        self,
        strategy: evosax.algorithms.population_based.PopulationBasedAlgorithm,
        params: Any,  # struct.dataclass with strategy hyper-parameters
        problem: evosax.problems.problem.Problem,
        problem_state: Any, # struct.dataclass with problem state
        pop_size: int,
        num_generations: int,
        num_dims: int,
        bounds: Tuple[float, float] = (-5.0, 5.0),
        maximize: bool = False,
        initial_population: chex.Array = None,
        prng_impl: Optional[str] = None,
    ) -> None:
        self.strategy = strategy
        self.params = params
        self.problem = problem
        self.problem_state = problem_state
        self.pop_size = pop_size
        self.num_generations = num_generations
        self.num_dims = num_dims
        self.bounds = bounds
        self.maximize = maximize
        self.initial_population = initial_population
        self.prng_impl = prng_impl


    def run_once(self,
                 key: chex.Array,
                 unroll_factor: int = 1,
                 compile: bool = True
                ) -> Dict[str, Any]:
        """Run one evolutionary experiment and return BenchmarkRunner-compatible results.

        Returns
        -------
        dict
            ``history`` : List[Dict] - per-generation stats
            ``summary`` : Dict       - final summary metrics
            ``timings`` : Dict       - wall-clock timing breakdown
        """
        start_time = time.perf_counter()

        # 1. Initialize the starting population and evaluate its fitness
        key, key_pop, key_eval = jax.random.split(key, 3)
        
        if self.initial_population is not None:
            population_init = self.initial_population
        else:
            keys = jax.random.split(key_pop, self.pop_size)
            population_init = jax.vmap(self.problem.sample)(keys)

        fitness_init, prob_state_init, _ = self.problem.eval(
            key_eval, population_init, self.problem_state
        )

        # Evosax algorithms natively minimize. If maximize is True, we must negate the fitness
        tell_fitness_init = -fitness_init if self.maximize else fitness_init

        # 2. Define the main evolutionary step for JAX to scan over
        def scan_step(carry, _):
            rng, state, p_state = carry
            rng, key_ask, key_eval_step, key_tell = jax.random.split(rng, 4)

            # Ask: Generate candidates
            population, state = self.strategy.ask(key_ask, state, self.params)
            
            # Eval: Assess candidates
            fitness, p_state, _ = self.problem.eval(key_eval_step, population, p_state)
            
            # Invert fitness if maximizing before passing back to evosax
            tell_fitness = -fitness if self.maximize else fitness

            # Tell: Update strategy internal tracking
            state, metrics = self.strategy.tell(key_tell, population, tell_fitness, state, self.params)

            # Correct the logged metrics to represent the true user-facing fitness 
            if self.maximize:
                metrics["best_fitness"] = -metrics["best_fitness"]
                metrics["best_fitness_in_generation"] = -metrics["best_fitness_in_generation"]

            return (rng, state, p_state), metrics

        # 3. Define the main execution block
        def run_loop(rng, pop_init, fit_init, p_state):
            rng, key_init = jax.random.split(rng)
            # Initialize population-based strategy
            state = self.strategy.init(key_init, pop_init, fit_init, self.params)
            
            carry = (rng, state, p_state)
            carry, metrics = jax.lax.scan(
                scan_step, 
                carry, 
                None, 
                length=self.num_generations, 
                unroll=unroll_factor
            )
            return carry[1], metrics

        # 4. Optional Compilation
        if compile:
            run_loop = jax.jit(run_loop)

        # 5. Execute 
        compile_start = time.perf_counter()
        final_state, metrics = run_loop(
            key, population_init, tell_fitness_init, prob_state_init
        )
        
        # Block until ready to ensure async GPU/TPU tasks complete before calculating timings
        jax.tree_util.tree_map(lambda x: x.block_until_ready(), final_state)
        end_time = time.perf_counter()

        # 6. Transform `evosax` Dict[str, Array] into the expected BenchmarkRunner format List[Dict]
        history = []
        for g in range(self.num_generations):
            gen_stats = {}
            for k, v in metrics.items():
                val = v[g]
                # Check if the metric is a scalar or an array
                if val.ndim == 0:
                    gen_stats[k] = val.item()
                else:
                    gen_stats[k] = val.tolist()
            history.append(gen_stats)

        # 7. Package results
        summary = {
            "best_fitness": history[-1].get("best_fitness", None) if history else None,
            "best_solution": self.strategy.get_best_solution(final_state).tolist(),
            "total_generations": self.num_generations,
            "pop_size": self.pop_size
        }

        timings = {
            "total_time": end_time - start_time,
            "run_time": end_time - compile_start,
        }

        return {
            "history": history,
            "summary": summary,
            "timings": timings
        }
def build_evosax_engine(
        strategy_name: str = "SimpleGA",
        *,
        evaluator: Optional[BaseEvaluator] = None,
        fitness_spec: Optional[str] = None,
        pop_size: int = 50,
        generations: int = 100,
        bounds: Tuple[float, float] = (-5.0, 5.0),
        maximize: bool = False,
        seed: int = 42,
        strategy_params: Optional[Dict[str, Any]] = None,
        initial_population: Any = None,
        prng_impl: Optional[str] = None,
        **kwargs: Any,
    ) -> EvosaxEngineAdapter:
        """Build an :class:`EvosaxEngineAdapter` from high-level specs.

        Parameters
        ----------
        strategy_name
            Name of the evosax strategy (``SimpleGA``, ``MR15_GA``,
            ``DifferentialEvolution``).
        evaluator
            A MalthusJAX :class:`BaseEvaluator` instance describing the
            fitness function.  If the object is a :class:`BBOBEvaluator` the
            underlying evosax problem is unwrapped automatically.  Support for
            other evaluator types is not yet implemented and will raise
            ``NotImplementedError``.
        fitness_spec
            Optional catalog-style spec that may override the configuration of a
            ``BBOBEvaluator`` when one is provided.  Has no effect for other
            evaluator types.
        pop_size
            Population size.
        generations
            Number of generations.
        bounds
            Search domain as ``(min, max)``.
        maximize
            If ``True``, flip the sign so the adapter reports fitness in
            maximisation convention (matching MalthusJAX default).
        seed
            Seed for the BBOB problem rotation/shift.
        strategy_params
            Optional dict of strategy-specific hyper-parameters that will be
            merged into ``strategy.default_params`` via ``.replace()``.

        Returns
        -------
        EvosaxEngineAdapter
            Ready to call ``.run_once(key)``.
        """
        if "num_generations" in kwargs:
            generations = int(kwargs.pop("num_generations"))

        if evaluator is None:
            raise ValueError("build_evosax_engine requires an evaluator argument")

        if fitness_spec is not None and isinstance(evaluator, BBOBEvaluator):
            from malthusjax.catalog import OperatorCatalog

            cat = OperatorCatalog()
            parsed_name, parsed_params = cat.parse_spec(fitness_spec)
            fn = parsed_params.get("fn_name", parsed_name)
            dims = parsed_params.get("dim", parsed_params.get("num_dims"))
            if dims is None:
                dims = evaluator.config.num_dims
            if "seed" in parsed_params:
                seed = parsed_params["seed"]
            if "maximize" in parsed_params:
                maximize = parsed_params["maximize"]
            evaluator = BBOBEvaluator.create(
                BBOBConfig(fn_name=fn, num_dims=dims, seed=seed, maximize=maximize)
            )

        if strategy_name not in EVOSAX_STRATEGIES:
            raise KeyError(f"Unknown evosax strategy '{strategy_name}'. Available: {list_strategies()}")

        rng = jr.PRNGKey(seed)

        if isinstance(evaluator, BBOBEvaluator):
            problem = evaluator.evosax_problem
            problem_state = evaluator.problem_state
            num_dims = evaluator.config.num_dims
        else:
            raise NotImplementedError(
                "Only BBOBEvaluator instances are currently supported by the "
                "evosax adapter; generic BaseEvaluator wrapping is TODO."
            )

        init_solution = jr.uniform(rng, (num_dims,), minval=bounds[0], maxval=bounds[1])

        strategy_cls = EVOSAX_STRATEGIES[strategy_name]
        strategy = strategy_cls(population_size=pop_size, solution=init_solution)

        params = strategy.default_params
        if strategy_params:
            params = params.replace(**strategy_params)

        return EvosaxEngineAdapter(
            strategy=strategy,
            params=params,
            problem=problem,
            problem_state=problem_state,
            pop_size=pop_size,
            num_generations=generations,
            num_dims=num_dims,
            bounds=bounds,
            maximize=maximize,
            initial_population=initial_population,
            prng_impl=prng_impl,
        )

In [23]:
import jax
import jax.random as jr
from malthusjax.core.fitness.bbob_evaluator import BBOBEvaluator, BBOBConfig

# 1. Setup the JAX random key
key = jr.PRNGKey(42)

# 2. Define the evaluator using the MalthusJAX BBOB config
print("Initializing BBOB Evaluator...")
evalr = BBOBEvaluator.create(
    BBOBConfig(fn_name="sphere", num_dims=10, seed=0, maximize=False)
)

# 3. Build the Evosax Engine
print("Building the Evosax Engine Adapter...")
adapter = build_evosax_engine(
    strategy_name="SimpleGA",
    evaluator=evalr,
    pop_size=100,
    generations=200,
    bounds=(-5.0, 5.0),
    maximize=False, # Set to True if your downstream task expects maximization
    seed=42
)

# 4. Run the optimization loop
print("Running the optimization...")
result = adapter.run_once(key)

print (result)

Initializing BBOB Evaluator...
Building the Evosax Engine Adapter...
Running the optimization...
{'history': [{'best_fitness': 91.39839172363281, 'best_fitness_in_generation': 91.39839172363281, 'best_solution': [0.7118982672691345, 2.8654510974884033, -6.401625633239746, 2.7756118774414062, 4.803638458251953, -0.22021673619747162, 0.03313625231385231, 2.0776782035827637, -4.509242057800293, 3.250014066696167], 'best_solution_in_generation': [0.7118982672691345, 2.8654510974884033, -6.401625633239746, 2.7756118774414062, 4.803638458251953, -0.22021673619747162, 0.03313625231385231, 2.0776782035827637, -4.509242057800293, 3.250014066696167], 'best_solution_norm': 10.758233070373535, 'generation_counter': 0}, {'best_fitness': 80.35845947265625, 'best_fitness_in_generation': 80.35845947265625, 'best_solution': [0.901817262172699, 3.6334011554718018, -5.148350238800049, 0.8376045823097229, 2.1830132007598877, 2.951697826385498, 2.484659433364868, 4.397035598754883, -2.3961241245269775, 1.3

In [ ]:
adapter